# Video Model Training on Kaggle

Trains the project's VAE or DiT with **HuggingFace Hub checkpoint sync** so progress survives Kaggle's 12-hour session limit.

## One-time setup

1. **Settings sidebar (right) -> Accelerator -> GPU T4 x2 / P100** (whichever is available).
2. **Settings -> Internet -> ON**.
3. **Settings -> Add-ons -> Secrets -> Add Secret**: name `HF_TOKEN`, value = your HuggingFace token with write access (https://huggingface.co/settings/tokens).
4. **Settings -> Add-ons -> Secrets -> Add Secret**: name `HF_REPO`, value = something like `your-username/video-model-ckpts` (doesn't need to exist — it'll be created automatically as a private repo).
5. **Settings -> Add-ons -> Secrets -> Add Secret** (optional): name `GIT_REPO`, value = `https://github.com/your-username/video-model.git`.

## Each session

Click **Save Version -> Save & Run All (Commit)** and close the tab. Kaggle runs the notebook for up to 12 hours in the background. The script pushes checkpoints every 10 minutes; even if Kaggle kills the session, you'll only lose the last few minutes.

Next session: open this notebook again and **Save & Run All** — it pulls existing checkpoints and resumes.

## 1. Configure (edit these once)

In [ ]:
# What to train. Train VAE FIRST, then switch to 'dit'.
MODE = "vae"               # "vae" or "dit"
MODEL_CONFIG = "configs/model/dit_micro.yaml"
ONLINE_LIMIT = None         # int to cap online clips per epoch, or None
SYNC_EVERY_MINUTES = 10     # how often to push checkpoints to HF Hub
CHECKPOINT_EVERY = 500      # override training config — small = lose less on timeout

# Read secrets — set these in the Kaggle UI (see top of notebook)
from kaggle_secrets import UserSecretsClient
_secrets = UserSecretsClient()
HF_TOKEN = _secrets.get_secret("HF_TOKEN")
HF_REPO  = _secrets.get_secret("HF_REPO")
try:
    GIT_REPO = _secrets.get_secret("GIT_REPO")
except Exception:
    GIT_REPO = None  # If not set, you'll need to upload the code as a Kaggle Dataset (see next cell)

print(f"MODE={MODE}, model_config={MODEL_CONFIG}")
print(f"HF_REPO={HF_REPO}")
print(f"GIT_REPO={GIT_REPO}")

## 2. Get the code

Two options:
- **GitHub** (preferred): set the `GIT_REPO` secret to your repo URL. The cell below clones it.
- **Kaggle Dataset**: zip the project, upload as a private dataset, then in the right sidebar -> Add Input -> attach it. The dataset will appear under `/kaggle/input/<dataset-name>/`.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
PROJECT_DIR = WORK / "video-model"

if GIT_REPO:
    if PROJECT_DIR.exists():
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", GIT_REPO, str(PROJECT_DIR)], check=True)
else:
    # Find the project in /kaggle/input/* (uploaded as a Kaggle Dataset)
    candidates = list(Path("/kaggle/input").glob("*/scripts/kaggle_train.py"))
    if not candidates:
        raise RuntimeError("No GIT_REPO secret AND no project dataset found in /kaggle/input/. "
                           "Either set the GIT_REPO secret or upload the project as a Kaggle dataset.")
    src = candidates[0].parent.parent
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(src, PROJECT_DIR)
    print(f"Copied project from {src} -> {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("CWD:", Path.cwd())
subprocess.run(["ls", "-la"], check=False)

## 3. Install dependencies

Kaggle's base image ships PyTorch + transformers already. We only install what's missing.

In [ ]:
# Install only what Kaggle doesn't already have. xformers is intentionally skipped.
%pip install -q accelerate safetensors omegaconf einops imageio imageio-ffmpeg lpips pytorch-fid huggingface-hub datasets torchcodec

In [ ]:
# Confirm GPU is visible
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. Train

Calls `scripts/kaggle_train.py` which:
- Pulls existing checkpoints from `HF_REPO`
- Starts background sync (every `SYNC_EVERY_MINUTES`)
- Runs training (auto-resumes from pulled checkpoints)
- Pushes a final time on exit (including if Kaggle kills the session)

In [ ]:
import os, subprocess, sys

# Export token so kaggle_train.py + HF libraries can use it without --hf_token in argv.
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

ckpt_dir = "checkpoints/vae" if MODE == "vae" else "checkpoints"

cmd = [
    sys.executable, "scripts/kaggle_train.py",
    "--mode", MODE,
    "--model_config", MODEL_CONFIG,
    "--hf_repo", HF_REPO,
    "--sync_every_minutes", str(SYNC_EVERY_MINUTES),
    "--checkpoint_dir", ckpt_dir,
    "--overrides",
    f"training.checkpoint_every={CHECKPOINT_EVERY}",
    f"training.checkpoint_dir={ckpt_dir}",
]
if ONLINE_LIMIT is not None:
    cmd += ["--limit", str(ONLINE_LIMIT)]

print("Running:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=False)

## 5. (Optional) Manual final push

Normally `kaggle_train.py` pushes on exit. If you want to force one now (e.g. after stopping training manually), run this:

In [ ]:
import os, subprocess, sys
os.environ["HF_TOKEN"] = HF_TOKEN
subprocess.run([
    sys.executable, "scripts/hf_sync.py", "push",
    "--local_dir", "checkpoints",
    "--repo_id", HF_REPO,
], check=False)